# 10 — CTGAN Data Augmentation Lab

Thử CTGAN trên Cleveland theo hướng leakage-safe: khóa test trước, CTGAN chỉ fit train, sinh x5/x10 qua nhiều seed, kiểm tra fidelity/memorization rồi benchmark trên test thật và hospital severe. Synthetic data không thay thế external validation.

In [ ]:
!pip install -q gdown sdv xgboost lightgbm

In [ ]:
import time
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import ks_2samp
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
FULL_EXPERIMENT = False  # True: 5 seeds x 500 epochs; nen dung GPU
SEEDS = [42, 52, 62, 72, 82] if FULL_EXPERIMENT else [42, 52, 62]
CTGAN_EPOCHS = 500 if FULL_EXPERIMENT else 300
MULTIPLIERS = [5, 10]
USE_GPU = torch.cuda.is_available()
print({'GPU': USE_GPU, 'seeds': SEEDS, 'epochs': CTGAN_EPOCHS})

## 1. Load và khóa test thật

Không sinh dữ liệu trước khi split. `target_original` được chuyển thành nhị phân: 0 là không bệnh, lớn hơn 0 là có bệnh.

In [ ]:
FILE_ID = '1YTzUy_RreXqnM5fqMR0hOLZPeXOvYoju'
DATA_PATH = Path('/content/cleveland.csv')
gdown.download(id=FILE_ID, output=str(DATA_PATH), quiet=False)

FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = [c for c in FEATURES if c not in NUMERICAL_FEATURES]
ALL_COLUMNS = FEATURES + ['target']

df = pd.read_csv(DATA_PATH, header=None, names=FEATURES+['target_original'], na_values=['?'])
df['target'] = (pd.to_numeric(df['target_original'], errors='coerce') > 0).astype(int)
for column in FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df[ALL_COLUMNS].drop_duplicates().reset_index(drop=True)
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df['target'], random_state=RANDOM_STATE)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
assert len(df) == 303 and train_df.shape[1] == 14
print('Train:', train_df.shape, 'Test:', test_df.shape)
display(train_df.head())

## 2. Fit CTGAN và sinh x5/x10

CTGAN học đồng thời 13 feature và target. Mỗi seed fit một generator độc lập. x5/x10 là tổng kích thước cuối cùng; dữ liệu thật luôn được giữ trong train mở rộng.

In [ ]:
CLINICAL_BOUNDS = {
    'age':(18,100), 'trestbps':(60,260), 'chol':(80,800),
    'thalach':(40,240), 'oldpeak':(0.0,10.0)}
INTEGER_NUMERICAL = ['age','trestbps','chol','thalach']

def make_metadata(frame):
    metadata = Metadata.detect_from_dataframe(data=frame, table_name='patients')
    for column in CATEGORICAL_FEATURES + ['target']:
        metadata.update_column(column_name=column, sdtype='categorical')
    for column in NUMERICAL_FEATURES:
        metadata.update_column(column_name=column, sdtype='numerical')
    metadata.validate()
    return metadata

def nearest_allowed(series, allowed):
    allowed = np.asarray(sorted(allowed), dtype=float)
    values = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    valid = ~np.isnan(values)
    values[valid] = allowed[np.abs(values[valid,None]-allowed[None,:]).argmin(axis=1)]
    return values

def enforce_domains(sample, reference):
    result = sample.copy()
    for column, (low, high) in CLINICAL_BOUNDS.items():
        result[column] = pd.to_numeric(result[column], errors='coerce').clip(low, high)
    result[INTEGER_NUMERICAL] = result[INTEGER_NUMERICAL].round()
    result['oldpeak'] = result['oldpeak'].round(1)
    for column in CATEGORICAL_FEATURES:
        result[column] = nearest_allowed(result[column], reference[column].dropna().unique())
    result['target'] = nearest_allowed(result['target'], [0,1]).astype(int)
    return result[ALL_COLUMNS]

def fit_ctgan(real_train, seed):
    np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    metadata = make_metadata(real_train)
    synthesizer = CTGANSynthesizer(
        metadata=metadata, epochs=CTGAN_EPOCHS, batch_size=50, pac=10,
        verbose=False, cuda=USE_GPU, enforce_min_max_values=True, enforce_rounding=True)
    started = time.perf_counter()
    synthesizer.fit(real_train)
    fit_seconds = time.perf_counter() - started
    return synthesizer, fit_seconds

def sample_expanded_train(synthesizer, real_train, multiplier, seed):
    n_synthetic = len(real_train) * (multiplier - 1)
    synthetic = enforce_domains(synthesizer.sample(num_rows=n_synthetic), real_train)
    synthetic['data_origin'] = 'synthetic'
    real = real_train.copy(); real['data_origin'] = 'real'
    expanded = pd.concat([real, synthetic], ignore_index=True).sample(
        frac=1, random_state=seed).reset_index(drop=True)
    return expanded

ctgan_datasets, generator_runs = {}, []
for seed in SEEDS:
    synthesizer, seconds = fit_ctgan(train_df, seed)
    for multiplier in MULTIPLIERS:
        name = f'ctgan_x{multiplier}_seed{seed}'
        ctgan_datasets[name] = sample_expanded_train(
            synthesizer, train_df, multiplier, seed)
        generator_runs.append({'dataset':name, 'seed':seed, 'multiplier':multiplier,
                               'rows':len(ctgan_datasets[name]), 'fit_seconds':seconds})
        print('Sampled', name)
    print('Fitted CTGAN seed', seed, 'in', round(seconds,1), 'seconds')
display(pd.DataFrame(generator_runs).round(2))

## 3. Fidelity và memorization

KS/TV càng thấp càng giống. Exact-match phải thấp. Ngoài ra so sánh correlation matrix để phát hiện CTGAN giữ marginal distribution nhưng làm mất quan hệ giữa feature.

In [ ]:
def total_variation(real, synthetic):
    categories = sorted(set(real.dropna().unique()) | set(synthetic.dropna().unique()))
    p = real.value_counts(normalize=True).reindex(categories, fill_value=0)
    q = synthetic.value_counts(normalize=True).reindex(categories, fill_value=0)
    return float(0.5 * np.abs(p-q).sum())

def evaluate_synthetic(name, expanded):
    synthetic = expanded[expanded['data_origin']=='synthetic'][ALL_COLUMNS]
    numeric_ks = np.mean([ks_2samp(train_df[c].dropna(), synthetic[c].dropna()).statistic
                          for c in NUMERICAL_FEATURES])
    categorical_tv = np.mean([total_variation(train_df[c], synthetic[c])
                               for c in CATEGORICAL_FEATURES+['target']])
    exact = synthetic.merge(train_df.drop_duplicates(), how='inner').shape[0] / len(synthetic)
    real_corr = train_df[NUMERICAL_FEATURES+['target']].corr().fillna(0)
    synth_corr = synthetic[NUMERICAL_FEATURES+['target']].corr().fillna(0)
    corr_mae = float(np.abs(real_corr-synth_corr).to_numpy().mean())
    return {'dataset':name, 'mean_numeric_ks':numeric_ks,
            'mean_categorical_tv':categorical_tv, 'correlation_mae':corr_mae,
            'exact_match_rate':exact,
            'synthetic_positive_rate':synthetic['target'].mean()}

quality_results = pd.DataFrame([evaluate_synthetic(name, data)
                                for name, data in ctgan_datasets.items()])
display(quality_results.sort_values(['mean_numeric_ks','mean_categorical_tv']).round(4))

fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.boxplot(data=quality_results.assign(multiplier=quality_results.dataset.str.extract(r'x(\d+)')[0]),
            x='multiplier', y='mean_numeric_ks', ax=axes[0])
sns.boxplot(data=quality_results.assign(multiplier=quality_results.dataset.str.extract(r'x(\d+)')[0]),
            x='multiplier', y='correlation_mae', ax=axes[1])
axes[0].set_title('Numeric fidelity across seeds'); axes[1].set_title('Correlation error');
plt.show()

## 4. Tạo severe hospital test

Giữ nguyên label; chỉ làm bẩn input bằng missing, outlier, làm tròn, sai categorical code và hospital shift.

In [ ]:
def make_severe_test(frame, seed):
    result = frame.copy().reset_index(drop=True)
    local_rng = np.random.default_rng(seed)
    result[FEATURES] = result[FEATURES].mask(
        local_rng.random((len(result),len(FEATURES))) < 0.15)
    for column in NUMERICAL_FEATURES:
        mask = local_rng.random(len(result)) < 0.05
        result.loc[mask,column] *= local_rng.choice([0.1,0.5,1.5,10.0], mask.sum())
    for column, step in {'trestbps':5,'chol':10,'thalach':5,'oldpeak':0.5}.items():
        mask = local_rng.random(len(result)) < 0.20
        result.loc[mask,column] = (result.loc[mask,column]/step).round()*step
    for column in CATEGORICAL_FEATURES:
        result.loc[local_rng.random(len(result)) < 0.02,column] = 99
    shifted = local_rng.random(len(result)) < 1.0
    result.loc[shifted,'age'] += 3; result.loc[shifted,'trestbps'] += 5
    return result

test_sets = {'clean':test_df.copy(), 'severe':make_severe_test(test_df, 999)}
print('Severe missing cells:', int(test_sets['severe'][FEATURES].isna().sum().sum()))

## 5. Downstream utility qua nhiều seed

Mỗi CTGAN dataset được dùng train cùng 5 model. Baseline real được chạy một lần. Bảng tổng hợp theo multiplier phản ánh biến thiên giữa generator seeds.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),
                        ('scaler',MinMaxScaler())])
    categorical = Pipeline([('imputer',SimpleImputer(strategy='most_frequent',add_indicator=True)),
                            ('encoder',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num',numeric,NUMERICAL_FEATURES),
                              ('cat',categorical,CATEGORICAL_FEATURES)])

MODELS = {
 'Logistic Regression':LogisticRegression(max_iter=1500,random_state=RANDOM_STATE),
 'Random Forest':RandomForestClassifier(n_estimators=400,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1),
 'Extra Trees':ExtraTreesClassifier(n_estimators=400,min_samples_leaf=2,random_state=RANDOM_STATE,n_jobs=-1),
 'XGBoost':XGBClassifier(n_estimators=300,max_depth=3,learning_rate=.03,subsample=.8,
                         colsample_bytree=.8,eval_metric='logloss',random_state=RANDOM_STATE,n_jobs=-1),
 'LightGBM':LGBMClassifier(n_estimators=300,num_leaves=15,learning_rate=.03,
                           random_state=RANDOM_STATE,verbosity=-1,n_jobs=-1)}

training_sets = {'real':train_df.copy()}
training_sets.update({name:data.drop(columns='data_origin') for name,data in ctgan_datasets.items()})
records = []
for dataset_name, training in training_sets.items():
    for model_name, classifier in MODELS.items():
        pipeline = Pipeline([('preprocessor',make_preprocessor()),('classifier',classifier)])
        pipeline.fit(training[FEATURES],training['target'])
        for test_name, testing in test_sets.items():
            probability = pipeline.predict_proba(testing[FEATURES])[:,1]
            prediction = (probability >= .5).astype(int)
            tn,fp,fn,tp = confusion_matrix(testing['target'],prediction,labels=[0,1]).ravel()
            records.append({'train_set':dataset_name,'model':model_name,'test_set':test_name,
                'accuracy':accuracy_score(testing['target'],prediction),
                'precision':precision_score(testing['target'],prediction,zero_division=0),
                'recall':recall_score(testing['target'],prediction,zero_division=0),
                'specificity':tn/(tn+fp) if tn+fp else np.nan,
                'false_negative_rate':fn/(fn+tp) if fn+tp else np.nan,
                'f1':f1_score(testing['target'],prediction,zero_division=0),
                'roc_auc':roc_auc_score(testing['target'],probability),
                'brier':brier_score_loss(testing['target'],probability)})
results = pd.DataFrame(records)
results['multiplier'] = results['train_set'].str.extract(r'x(\d+)')[0].fillna('real')
display(results.sort_values(['test_set','roc_auc'],ascending=[True,False]).round(4))

In [ ]:
ctgan_only = results[results['train_set']!='real']
summary = ctgan_only.groupby(['multiplier','model','test_set']).agg(
    roc_auc_mean=('roc_auc','mean'),roc_auc_std=('roc_auc','std'),
    recall_mean=('recall','mean'),recall_std=('recall','std'),recall_min=('recall','min'),
    f1_mean=('f1','mean'),brier_mean=('brier','mean')).reset_index()
display(summary.sort_values(['test_set','roc_auc_mean'],ascending=[True,False]).round(4))

clean = summary[summary.test_set=='clean'].drop(columns='test_set').rename(
    columns={'roc_auc_mean':'clean_auc','recall_mean':'clean_recall','brier_mean':'clean_brier'})
severe = summary[summary.test_set=='severe'].drop(columns='test_set').rename(
    columns={'roc_auc_mean':'severe_auc','recall_mean':'severe_recall','brier_mean':'severe_brier'})
comparison = clean.merge(severe,on=['multiplier','model'],suffixes=('_clean','_severe'))
comparison['auc_drop'] = comparison['clean_auc']-comparison['severe_auc']
comparison['recall_drop'] = comparison['clean_recall']-comparison['severe_recall']
comparison = comparison.sort_values(['severe_auc','auc_drop','severe_recall'],ascending=[False,True,False])
display(comparison.round(4))
print('CTGAN ROBUSTNESS LEADER')
display(comparison.head(1))

## 6. Quy tắc kết luận

CTGAN chỉ được giữ nếu thắng Gaussian Copula/real baseline qua nhiều seed, exact-match thấp, correlation error chấp nhận được, Recall worst-case không xấu và Brier không tăng rõ rệt. Dataset chỉ có khoảng 242 train rows nên kết quả CTGAN có thể dao động mạnh. Notebook không xuất model hoặc synthetic CSV.